In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F

# Use UTC for source timestamps ending in Z
spark.conf.set("spark.sql.session.timeZone", "UTC")

# READ CSV FROM SILVER LAKEHOUSE
df_procedures = spark.read.csv(
    "Files/procedures.csv", header=True
)

In [ ]:
display(df_procedures)

In [ ]:
#TRANSFORMATIONS
df_procedures_transformed = (
    df_procedures

    # Convert START and STOP into timestamps
    .withColumn(
        "START",
        F.to_timestamp(
            "START",
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )
    .withColumn(
        "STOP",
        F.to_timestamp(
            "STOP",
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )

    # Extract date from timestamp
    .withColumn(
        "PROCEDURE_DATE",
        F.to_date("START")
    )

    # Calculate procedure duration in minutes
    .withColumn(
        "PROCEDURE_DURATION_MINUTES",
        F.round(
            (
                F.unix_timestamp("STOP")
                - F.unix_timestamp("START")
            ) / 60,
            2
        )
    )

    # Convert cost to numeric decimal
    .withColumn(
        "BASE_COST",
        F.col("BASE_COST").cast("decimal(18,2)")
    )

    # Preserve medical codes as strings
    .withColumn(
        "CODE",
        F.col("CODE").cast("string")
    )
    .withColumn(
        "REASONCODE",
        F.col("REASONCODE").cast("string")
    )

    # Clean description
    .withColumn(
        "DESCRIPTION",
        F.trim("DESCRIPTION")
    )

    # Create broad procedure categories
    .withColumn(
        "PROCEDURE_TYPE",
        F.when(
            F.lower(F.col("DESCRIPTION")).contains(
                "screening"
            ),
            "Screening"
        )
        .when(
            F.lower(F.col("DESCRIPTION")).contains(
                "assessment"
            ),
            "Assessment"
        )
        .when(
            F.lower(F.col("DESCRIPTION")).rlike(
                "therapy|dialysis|chemotherapy|radiation"
            ),
            "Therapy / Treatment"
        )
        .when(
            F.lower(F.col("DESCRIPTION")).rlike(
                "diagnostic|colonoscopy|bronchoscopy|x-ray"
            ),
            "Diagnostic"
        )
        .otherwise("Other")
    )
)


In [ ]:
display(df_procedures_transformed)

In [ ]:
path = spark.conf.get('spark.executorEnv.HC_Goldpath')
display(path)

In [ ]:

(
    df_procedures_transformed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "Healthcare_DE_Project.HC_Gold.dbo.DIM_Procedures"
    )
)
